In [1]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd().parent

data_path = (
    project_root
    / "data"
    / "processed"
    / "vic_demand_2024-07_to_2026-06.parquet"
)

df = pd.read_parquet(data_path)

df.head()

,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,VIC1,2024-07-01 00:05:00,5404.13,281.32,TRADE
1,VIC1,2024-07-01 00:10:00,5346.60,275.61,TRADE
2,VIC1,2024-07-01 00:15:00,5240.84,270.98,TRADE
3,VIC1,2024-07-01 00:20:00,5239.06,270.98,TRADE
4,VIC1,2024-07-01 00:25:00,5226.10,270.98,TRADE


In [2]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Start: {df['SETTLEMENTDATE'].min()}")
print(f"End: {df['SETTLEMENTDATE'].max()}")

print()
print("Regions:", df["REGION"].unique())
print("Period types:", df["PERIODTYPE"].unique())

print()
print("Missing values:")
print(df.isna().sum())

print()
print("Duplicate timestamps:", df["SETTLEMENTDATE"].duplicated().sum())

Rows: 210,240
Columns: 5
Start: 2024-07-01 00:05:00
End: 2026-07-01 00:00:00

Regions: <ArrowStringArray>
['VIC1']
Length: 1, dtype: str
Period types: <ArrowStringArray>
['TRADE']
Length: 1, dtype: str

Missing values:
REGION            0
SETTLEMENTDATE    0
TOTALDEMAND       0
RRP               0
PERIODTYPE        0
dtype: int64

Duplicate timestamps: 0


In [3]:
interval_counts = (
    df["SETTLEMENTDATE"]
    .sort_values()
    .diff()
    .value_counts()
)

interval_counts.head(10)

SETTLEMENTDATE
0 days 00:05:00    210239
Name: count, dtype: int64

In [4]:
expected_timestamps = pd.date_range(
    start=df["SETTLEMENTDATE"].min(),
    end=df["SETTLEMENTDATE"].max(),
    freq="5min",
)

actual_timestamps = pd.DatetimeIndex(
    df["SETTLEMENTDATE"]
)

missing_timestamps = expected_timestamps.difference(actual_timestamps)
unexpected_timestamps = actual_timestamps.difference(expected_timestamps)

print(f"Expected timestamps: {len(expected_timestamps):,}")
print(f"Actual timestamps:   {len(actual_timestamps):,}")
print(f"Missing timestamps:  {len(missing_timestamps):,}")
print(f"Unexpected timestamps: {len(unexpected_timestamps):,}")

if len(missing_timestamps) > 0:
    print("\nFirst missing timestamps:")
    print(missing_timestamps[:10])

Expected timestamps: 210,240
Actual timestamps:   210,240
Missing timestamps:  0
Unexpected timestamps: 0


In [6]:
df[["TOTALDEMAND", "RRP"]].describe()

print("Demand <= 0:", (df["TOTALDEMAND"] <= 0).sum())
print("Negative prices:", (df["RRP"] < 0).sum())

print()
print("Lowest demand observations:")
display(
    df.nsmallest(5, "TOTALDEMAND")[
        ["SETTLEMENTDATE", "TOTALDEMAND", "RRP"]
    ]
)

print()
print("Highest demand observations:")
display(
    df.nlargest(5, "TOTALDEMAND")[
        ["SETTLEMENTDATE", "TOTALDEMAND", "RRP"]
    ]
)

Demand <= 0: 0
Negative prices: 51798

Lowest demand observations:


,SETTLEMENTDATE,TOTALDEMAND,RRP
156827,2025-12-27 13:00:00,1224.77,-105.45
156828,2025-12-27 13:05:00,1230.24,-199.97
156822,2025-12-27 12:35:00,1238.67,-133.79
156824,2025-12-27 12:45:00,1241.53,-165.65
156826,2025-12-27 12:55:00,1246.37,-170.72



Highest demand observations:


,SETTLEMENTDATE,TOTALDEMAND,RRP
165815,2026-01-27 18:00:00,10783.74,1074.22
165816,2026-01-27 18:05:00,10750.24,768.90
165814,2026-01-27 17:55:00,10748.28,1703.80
165813,2026-01-27 17:50:00,10658.44,3026.01
165812,2026-01-27 17:45:00,10600.71,1937.23


In [7]:
monthly_counts = (
    df.assign(
        month=df["SETTLEMENTDATE"].dt.to_period("M")
    )
    .groupby("month")
    .size()
)

monthly_counts

month
2024-07    8927
2024-08    8928
2024-09    8640
2024-10    8928
2024-11    8640
2024-12    8928
2025-01    8928
2025-02    8064
2025-03    8928
2025-04    8640
2025-05    8928
2025-06    8640
2025-07    8928
2025-08    8928
2025-09    8640
2025-10    8928
2025-11    8640
2025-12    8928
2026-01    8928
2026-02    8064
2026-03    8928
2026-04    8640
2026-05    8928
2026-06    8640
2026-07       1
Freq: M, dtype: int64

## Historical dataset validation

The processed dataset combines monthly AEMO price and demand files for
Victoria from July 2024 through June 2026.

Validation checks assess:

- temporal coverage;
- five-minute interval consistency;
- missing and duplicate timestamps;
- missing observations;
- regional and period-type consistency; and
- plausible ranges for demand and wholesale electricity prices.

These checks are performed before feature engineering or model development
to ensure that forecasting results are not driven by unnoticed data-quality
problems.